# 第20章：高级编程 - 声明式编程与异常处理
# Chapter 20: Declarative Programming & Exception Handling

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

- 理解声明式编程 (Declarative Programming) 的核心思想
- 了解SQL作为声明式语言的特点
- 了解Prolog的基本概念：事实(Facts)、规则(Rules)、查询(Queries)
- 掌握文件处理：文本文件和随机文件
- 掌握异常处理 (Exception Handling) 的 TRY...EXCEPT 结构
- 了解函数式编程 (Functional Programming) 的基本思想

---
## 1. 声明式编程 (Declarative Programming)

### 1.1 核心思想

**声明式编程 (Declarative Programming)** 的核心是：**告诉计算机"要什么" (WHAT)，而不是"怎么做" (HOW)**。

> **类比**：你去餐厅点餐，你说"我要一份宫保鸡丁"（声明式），而不是"先把鸡胸肉切丁，然后把花生米炒香，再把辣椒切段..."（命令式）。厨师知道怎么做，你只需要说你要什么。

### 1.2 为什么需要声明式编程？

1. **更高的抽象层次**：不关心执行细节，专注于问题本身
2. **更接近人类思维**：描述问题比描述步骤更自然
3. **更少的bug**：不需要管理状态和控制流，减少出错机会
4. **可优化性**：底层引擎可以自动选择最优的执行策略

### 1.3 声明式编程的两大分支

| 分支 | 英文 | 代表语言 | 核心 |
|------|------|----------|------|
| 逻辑式编程 | Logic Programming | Prolog | 基于逻辑推理 |
| 函数式编程 | Functional Programming | Haskell, Lisp | 基于数学函数 |

---
## 2. SQL - 声明式语言的典范

### 2.1 SQL是什么？

**SQL (Structured Query Language)** 是最广泛使用的声明式语言，用于操作关系数据库。

### 2.2 SQL为什么是声明式的？

看这个SQL查询：
```sql
SELECT name, age FROM students WHERE grade = 'A' ORDER BY age;
```

你只说了"我要什么"：
- 要 name 和 age 两列
- 来自 students 表
- 条件是 grade = 'A'
- 按 age 排序

你**没有**说怎么找：数据库引擎自动决定用哪个索引、怎么扫描、怎么排序。

In [ ]:
# 用Python模拟SQL查询 - 对比命令式和声明式
import sqlite3

# 创建内存数据库
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 创建表
cursor.execute('''
    CREATE TABLE students (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER,
        grade TEXT,
        score REAL
    )
''')

# 插入数据
students_data = [
    (1, '张三', 17, 'A', 92.5),
    (2, '李四', 16, 'B', 78.0),
    (3, '王五', 17, 'A', 95.0),
    (4, '赵六', 18, 'C', 65.0),
    (5, '钱七', 16, 'A', 88.0),
    (6, '孙八', 17, 'B', 82.0),
    (7, '周九', 18, 'A', 90.0),
    (8, '吴十', 16, 'D', 55.0),
]
cursor.executemany('INSERT INTO students VALUES (?,?,?,?,?)', students_data)
conn.commit()

print('=== 声明式 vs 命令式对比 ===')
print()

# 声明式：SQL
print('--- 声明式 (SQL) ---')
sql = "SELECT name, age, score FROM students WHERE grade = 'A' ORDER BY score DESC"
print(f'SQL: {sql}')
print('结果：')
for row in cursor.execute(sql):
    print(f'  {row[0]}, {row[1]}岁, {row[2]}分')
print()

# 命令式：Python
print('--- 命令式 (Python) ---')
print('步骤：遍历 -> 筛选 -> 排序 -> 输出')
filtered = []
for student in students_data:
    if student[3] == 'A':              # 筛选
        filtered.append(student)
filtered.sort(key=lambda x: x[4], reverse=True)  # 排序
print('结果：')
for s in filtered:
    print(f'  {s[1]}, {s[2]}岁, {s[4]}分')
print()

# 更多SQL示例
print('--- 更多SQL声明式查询 ---')
print()

queries = [
    ('统计每个等级的人数', 'SELECT grade, COUNT(*) as count FROM students GROUP BY grade ORDER BY grade'),
    ('平均分最高的等级', 'SELECT grade, AVG(score) as avg_score FROM students GROUP BY grade ORDER BY avg_score DESC LIMIT 1'),
    ('年龄最大的学生', 'SELECT name, age FROM students WHERE age = (SELECT MAX(age) FROM students)'),
]

for desc, sql in queries:
    print(f'{desc}:')
    print(f'  SQL: {sql}')
    for row in cursor.execute(sql):
        print(f'  结果: {row}')
    print()

conn.close()



In [ ]:
# 可视化：声明式 vs 命令式思维方式对比
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# === 左图：命令式思维 ===
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('命令式思维 (Imperative)\n"HOW - 怎么做"', fontsize=13, fontweight='bold', color='#D32F2F')

steps = [
    (5, 11, '开始', '#FFCDD2'),
    (5, 9.5, '1. 创建空列表 result=[]', '#EF9A9A'),
    (5, 8, '2. 遍历每条记录', '#E57373'),
    (5, 6.5, '3. 检查 grade=="A" ?', '#EF5350'),
    (5, 5, '4. 如果是，添加到result', '#F44336'),
    (5, 3.5, '5. 按score排序result', '#E53935'),
    (5, 2, '6. 输出结果', '#D32F2F'),
    (5, 0.5, '结束', '#FFCDD2'),
]

for i, (x, y, text, color) in enumerate(steps):
    box = mpatches.FancyBboxPatch((x-3, y-0.4), 6, 0.8, boxstyle='round,pad=0.15',
        facecolor=color, edgecolor='#B71C1C', linewidth=1.5, alpha=0.8)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(steps) - 1:
        ax.annotate('', xy=(5, steps[i+1][1]+0.4), xytext=(5, y-0.4),
            arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5))

ax.text(5, -0.5, '程序员必须精确控制每一步', ha='center', fontsize=10, color='red')

# === 右图：声明式思维 ===
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('声明式思维 (Declarative)\n"WHAT - 要什么"', fontsize=13, fontweight='bold', color='#2E7D32')

# 只有一个大框
big_box = mpatches.FancyBboxPatch((1, 4), 8, 5, boxstyle='round,pad=0.3',
    facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=3)
ax.add_patch(big_box)

ax.text(5, 8, 'SQL 查询', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ax.text(5, 7, 'SELECT name, score', ha='center', fontsize=11, family='monospace')
ax.text(5, 6.3, 'FROM students', ha='center', fontsize=11, family='monospace')
ax.text(5, 5.6, "WHERE grade = 'A'", ha='center', fontsize=11, family='monospace')
ax.text(5, 4.9, 'ORDER BY score DESC', ha='center', fontsize=11, family='monospace')

# 数据库引擎
engine_box = mpatches.FancyBboxPatch((2, 1), 6, 2, boxstyle='round,pad=0.2',
    facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(engine_box)
ax.text(5, 2.3, '数据库引擎 (自动优化)', ha='center', fontsize=11, fontweight='bold', color='#F57F17')
ax.text(5, 1.6, '自动选择索引、执行计划...', ha='center', fontsize=9, color='#795548')

ax.annotate('', xy=(5, 3), xytext=(5, 4),
    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# 结果
ax.text(5, 10.5, '程序员只描述需求', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ax.text(5, 0.3, '引擎自动决定最优执行方式', ha='center', fontsize=10, color='green')

plt.tight_layout()


---
## 3. Prolog简介 - 逻辑式编程 (Logic Programming)

### 3.1 什么是Prolog？

**Prolog (Programming in Logic)** 是一种逻辑式编程语言，基于形式逻辑。它的程序由三部分组成：

1. **事实 (Facts)**：已知的真命题
2. **规则 (Rules)**：推理规则（如果...那么...）
3. **查询 (Queries)**：要回答的问题

### 3.2 为什么了解Prolog很重要？

1. 它代表了一种完全不同的编程思维
2. 是声明式编程的典型代表
3. 在AI、专家系统、自然语言处理中有重要应用
4. Cambridge A-Level考试要求了解

### 3.3 Prolog语法示例

```prolog
% 事实 (Facts)
parent(tom, bob).       % tom是bob的父母
parent(tom, liz).       % tom是liz的父母
parent(bob, ann).       % bob是ann的父母
male(tom).
male(bob).
female(liz).
female(ann).

% 规则 (Rules)
father(X, Y) :- parent(X, Y), male(X).    % X是Y的父亲，如果X是Y的父母且X是男性
grandfather(X, Y) :- father(X, Z), parent(Z, Y).  % X是Y的祖父

% 查询 (Queries)
?- father(tom, bob).      % tom是bob的父亲吗？ -> yes
?- grandfather(tom, ann).  % tom是ann的祖父吗？ -> yes
```

In [ ]:
# 用Python模拟Prolog风格的逻辑推理

class PrologEngine:
    """
    简易Prolog引擎 - 用Python模拟逻辑式编程
    展示事实(Facts)、规则(Rules)和查询(Queries)
    """
    
    def __init__(self):
        self.facts = {}    # 存储事实
        self.rules = {}    # 存储规则
    
    def add_fact(self, predicate, *args):
        """添加事实 (Fact)"""
        if predicate not in self.facts:
            self.facts[predicate] = []
        self.facts[predicate].append(args)
    
    def add_rule(self, name, func):
        """添加规则 (Rule)"""
        self.rules[name] = func
    
    def query_fact(self, predicate, *args):
        """查询事实"""
        if predicate not in self.facts:
            return False
        return args in self.facts[predicate]
    
    def query_all(self, predicate, position=None, value=None):
        """查询所有匹配的事实"""
        if predicate not in self.facts:
            return []
        if position is not None and value is not None:
            return [f for f in self.facts[predicate] if f[position] == value]
        return self.facts[predicate]
    
    def query_rule(self, rule_name, *args):
        """查询规则"""
        if rule_name in self.rules:
            return self.rules[rule_name](self, *args)
        return False


# ============ 建立知识库 ============
engine = PrologEngine()

# 事实 (Facts) - 类似Prolog的 parent(tom, bob).
print('=== 建立知识库（事实 Facts）===')

# parent(X, Y) 表示 X 是 Y 的父母
family_facts = [
    ('parent', '爷爷', '爸爸'),
    ('parent', '爷爷', '叔叔'),
    ('parent', '奶奶', '爸爸'),
    ('parent', '奶奶', '叔叔'),
    ('parent', '爸爸', '我'),
    ('parent', '爸爸', '妹妹'),
    ('parent', '妈妈', '我'),
    ('parent', '妈妈', '妹妹'),
    ('parent', '叔叔', '堂弟'),
]

for pred, *args in family_facts:
    engine.add_fact(pred, *args)
    print(f'  事实: parent({args[0]}, {args[1]}).  % {args[0]}是{args[1]}的父母')

# 性别事实
for person in ['爷爷', '爸爸', '叔叔', '我', '堂弟']:
    engine.add_fact('male', person)
for person in ['奶奶', '妈妈', '妹妹']:
    engine.add_fact('female', person)

print()

# 规则 (Rules) - 类似Prolog的 father(X,Y) :- parent(X,Y), male(X).
print('=== 定义规则 (Rules) ===')

def father_rule(engine, x, y):
    """father(X, Y) :- parent(X, Y), male(X)."""
    return engine.query_fact('parent', x, y) and engine.query_fact('male', x)

def mother_rule(engine, x, y):
    """mother(X, Y) :- parent(X, Y), female(X)."""
    return engine.query_fact('parent', x, y) and engine.query_fact('female', x)

def grandfather_rule(engine, x, y):
    """grandfather(X, Y) :- father(X, Z), parent(Z, Y)."""
    # 找到X的所有子女Z
    children = engine.query_all('parent', 0, x)
    for child_tuple in children:
        z = child_tuple[1]
        if engine.query_fact('male', x) and engine.query_fact('parent', z, y):
            return True
    return False

def sibling_rule(engine, x, y):
    """sibling(X, Y) :- parent(Z, X), parent(Z, Y), X \\= Y."""
    parents_of_x = [f[0] for f in engine.query_all('parent') if f[1] == x]
    parents_of_y = [f[0] for f in engine.query_all('parent') if f[1] == y]
    return x != y and bool(set(parents_of_x) & set(parents_of_y))

engine.add_rule('father', father_rule)
engine.add_rule('mother', mother_rule)
engine.add_rule('grandfather', grandfather_rule)
engine.add_rule('sibling', sibling_rule)

print('  规则: father(X, Y) :- parent(X, Y), male(X).')
print('  规则: mother(X, Y) :- parent(X, Y), female(X).')
print('  规则: grandfather(X, Y) :- father(X, Z), parent(Z, Y).')
print('  规则: sibling(X, Y) :- parent(Z, X), parent(Z, Y), X \\= Y.')
print()

# 查询 (Queries)
print('=== 查询 (Queries) ===')
queries = [
    ('father', '爸爸', '我', '爸爸是我的父亲吗？'),
    ('father', '妈妈', '我', '妈妈是我的父亲吗？'),
    ('mother', '妈妈', '我', '妈妈是我的母亲吗？'),
    ('grandfather', '爷爷', '我', '爷爷是我的祖父吗？'),
    ('grandfather', '爷爷', '堂弟', '爷爷是堂弟的祖父吗？'),
    ('sibling', '我', '妹妹', '我和妹妹是兄妹吗？'),
    ('sibling', '我', '堂弟', '我和堂弟是兄弟吗？'),
]

for rule, x, y, desc in queries:
    result = engine.query_rule(rule, x, y)
    status = 'yes' if result else 'no'
    print(f'  ?- {rule}({x}, {y}).  % {desc} -> {status}')

print()
print('注意：我们只定义了事实和规则，Prolog引擎自动进行逻辑推理！')


In [ ]:
# 可视化：家族关系图
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Prolog家族关系知识库可视化', fontsize=14, fontweight='bold', pad=15)

# 人物位置
people = {
    '爷爷': (4, 8.5, '#2196F3'),
    '奶奶': (8, 8.5, '#E91E63'),
    '爸爸': (2.5, 5.5, '#2196F3'),
    '妈妈': (5, 5.5, '#E91E63'),
    '叔叔': (9, 5.5, '#2196F3'),
    '我': (2, 2.5, '#2196F3'),
    '妹妹': (5, 2.5, '#E91E63'),
    '堂弟': (9, 2.5, '#2196F3'),
}

# 画人物
for name, (x, y, color) in people.items():
    circle = plt.Circle((x, y), 0.5, facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# 画关系线
parent_relations = [
    ('爷爷', '爸爸'), ('爷爷', '叔叔'),
    ('奶奶', '爸爸'), ('奶奶', '叔叔'),
    ('爸爸', '我'), ('爸爸', '妹妹'),
    ('妈妈', '我'), ('妈妈', '妹妹'),
    ('叔叔', '堂弟'),
]

for parent, child in parent_relations:
    px, py, _ = people[parent]
    cx, cy, _ = people[child]
    ax.annotate('', xy=(cx, cy+0.5), xytext=(px, py-0.5),
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

# 标注规则推导
ax.annotate('grandfather(爷爷, 我)\n= father(爷爷,爸爸)\n  + parent(爸爸,我)',
    xy=(2, 2.5), xytext=(0.2, 0.5),
    fontsize=8, color='#FF6F00', fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF8E1', edgecolor='#FF6F00'),
    arrowprops=dict(arrowstyle='->', color='#FF6F00', lw=1.5))

ax.annotate('sibling(我, 妹妹)\n= parent(爸爸,我)\n  + parent(爸爸,妹妹)',
    xy=(3.5, 2.5), xytext=(6.5, 1),
    fontsize=8, color='#4CAF50', fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', edgecolor='#4CAF50'),
    arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1.5))

# 图例
ax.text(11, 8.5, '图例:', fontsize=10, fontweight='bold')
blue_dot = plt.Circle((10.5, 7.8), 0.2, facecolor='#2196F3')
pink_dot = plt.Circle((10.5, 7.2), 0.2, facecolor='#E91E63')
ax.add_patch(blue_dot)
ax.add_patch(pink_dot)
ax.text(11, 7.8, '= male', fontsize=9, va='center')
ax.text(11, 7.2, '= female', fontsize=9, va='center')
ax.text(11, 6.5, '-> = parent', fontsize=9)

plt.tight_layout()


---
## 4. 文件处理 (File Handling)

### 4.1 为什么需要文件处理？

程序运行时的数据存储在内存中（RAM），程序关闭后数据就丢失了。**文件处理**让我们可以把数据**持久化**到磁盘上。

### 4.2 文件类型

| 类型 | 英文 | 特点 | 适用场景 |
|------|------|------|----------|
| 文本文件 | Text File | 人类可读，按行存储 | 日志、配置、CSV |
| 二进制文件 | Binary File | 非文本数据 | 图片、音频、视频 |
| 随机文件 | Random Access File | 可以直接跳转到任意位置 | 数据库、记录文件 |

### 4.3 文件操作模式

| 模式 | 含义 | 说明 |
|------|------|------|
| `'r'` | Read | 只读（文件必须存在） |
| `'w'` | Write | 写入（覆盖已有内容） |
| `'a'` | Append | 追加（在末尾添加） |
| `'r+'` | Read+Write | 读写 |
| `'b'` | Binary | 二进制模式（如`'rb'`） |

In [ ]:
# 文件处理示例
import os
import tempfile

# 使用临时目录避免文件残留
temp_dir = tempfile.mkdtemp()

# ============ 1. 文本文件写入 ============
print('=== 1. 文本文件写入 (Write) ===')
filepath = os.path.join(temp_dir, 'students.txt')

with open(filepath, 'w', encoding='utf-8') as f:
    f.write('学号,姓名,成绩\n')
    f.write('S001,张三,92\n')
    f.write('S002,李四,78\n')
    f.write('S003,王五,85\n')
    f.write('S004,赵六,65\n')

print(f'文件已写入: {filepath}')
print()

# ============ 2. 文本文件读取 ============
print('=== 2. 文本文件读取 (Read) ===')

# 方法1：read() - 读取全部内容
with open(filepath, 'r', encoding='utf-8') as f:
    content = f.read()
print('--- read() 读取全部 ---')
print(content)

# 方法2：readline() - 逐行读取
print('--- readline() 逐行读取 ---')
with open(filepath, 'r', encoding='utf-8') as f:
    line = f.readline()
    while line:
        print(f'  {repr(line.strip())}')
        line = f.readline()
print()

# 方法3：readlines() - 读取所有行到列表
print('--- readlines() 读取为列表 ---')
with open(filepath, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        print(f'  第{i}行: {line.strip()}')
print()

# ============ 3. 追加写入 ============
print('=== 3. 追加写入 (Append) ===')
with open(filepath, 'a', encoding='utf-8') as f:
    f.write('S005,钱七,90\n')
print('已追加一条记录')
print()

# ============ 4. 随机访问 (seek) ============
print('=== 4. 随机访问 (Random Access - seek) ===')
with open(filepath, 'r', encoding='utf-8') as f:
    print(f'当前位置: {f.tell()}')
    f.seek(0)  # 跳转到文件开头
    first_line = f.readline()
    print(f'第一行: {first_line.strip()}')
    pos = f.tell()
    print(f'读取后位置: {pos}')
    f.seek(0)  # 回到开头
    print(f'seek(0)后位置: {f.tell()}')

print()
print('关键概念：')
print('  seek(n) - 将文件指针移动到第n个字节')
print('  tell()  - 返回当前文件指针的位置')
print('  随机访问允许直接跳转到文件中任意位置读写')

# 清理
import shutil


In [ ]:
# 文件处理实用示例：CSV数据分析
import tempfile, os, csv

temp_dir = tempfile.mkdtemp()
csv_path = os.path.join(temp_dir, 'scores.csv')

# 写入CSV
data = [
    ['Name', 'Math', 'Physics', 'Chemistry'],
    ['Zhang San', 92, 88, 95],
    ['Li Si', 78, 85, 72],
    ['Wang Wu', 65, 58, 70],
    ['Zhao Liu', 45, 52, 48],
    ['Qian Qi', 88, 92, 90],
]

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(data)

print('=== CSV文件读取与分析 ===')
print()

# 读取CSV
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    print(f'表头: {header}')
    print()
    
    students = []
    for row in reader:
        name = row[0]
        scores = [int(x) for x in row[1:]]
        avg = sum(scores) / len(scores)
        students.append((name, scores, avg))
        print(f'  {name}: {scores}, 平均分={avg:.1f}')

print()

# 统计
all_averages = [s[2] for s in students]
print(f'全班平均分: {sum(all_averages)/len(all_averages):.1f}')
best = max(students, key=lambda s: s[2])
print(f'最高平均分: {best[0]} ({best[2]:.1f})')
worst = min(students, key=lambda s: s[2])
print(f'最低平均分: {worst[0]} ({worst[2]:.1f})')

import shutil


---
## 5. 异常处理 (Exception Handling)

### 5.1 什么是异常？

**异常 (Exception)** 是程序运行时发生的错误。如果不处理，程序会崩溃。

### 5.2 为什么需要异常处理？

现实世界中，错误是不可避免的：
- 用户输入了非数字字符
- 要读取的文件不存在
- 网络连接中断
- 除以零

**异常处理让程序能够"优雅地"处理错误**，而不是直接崩溃。就像汽车的安全气囊——碰撞不可避免，但我们可以减少伤害。

### 5.3 TRY...EXCEPT 结构

```
Cambridge伪代码:
TRY
    <可能出错的代码>
EXCEPT
    <错误处理代码>
ENDTRY

Python:
try:
    <可能出错的代码>
except <异常类型>:
    <错误处理代码>
```

In [ ]:
# 异常处理基础演示

print('=== 异常处理 (Exception Handling) 基础 ===')
print()

# ============ 1. 没有异常处理 vs 有异常处理 ============
print('--- 1. ZeroDivisionError ---')

# 没有处理：程序崩溃
# result = 10 / 0  # 取消注释会导致崩溃！

# 有异常处理：程序继续运行
try:
    result = 10 / 0
except ZeroDivisionError:
    print('  错误：不能除以零！')
    result = None
print(f'  程序继续运行，result = {result}')
print()

# ============ 2. 常见异常类型 ============
print('--- 2. 常见异常类型 ---')
print()

# ValueError
print('ValueError (值错误):')
try:
    number = int('hello')
except ValueError as e:
    print(f'  捕获到ValueError: {e}')
print()

# TypeError
print('TypeError (类型错误):')
try:
    result = '3' + 5
except TypeError as e:
    print(f'  捕获到TypeError: {e}')
print()

# IndexError
print('IndexError (索引错误):')
try:
    lst = [1, 2, 3]
    print(lst[10])
except IndexError as e:
    print(f'  捕获到IndexError: {e}')
print()

# KeyError
print('KeyError (键错误):')
try:
    d = {'name': '张三'}
    print(d['age'])
except KeyError as e:
    print(f'  捕获到KeyError: {e}')
print()

# FileNotFoundError
print('FileNotFoundError (文件未找到):')
try:
    with open('nonexistent_file.txt', 'r') as f:
        content = f.read()
except FileNotFoundError as e:
    print(f'  捕获到FileNotFoundError: {e}')
print()

print('每种异常都对应一种特定类型的运行时错误。')


In [ ]:
# 异常处理完整结构：try / except / else / finally

print('=== 完整的异常处理结构 ===')
print()
print('结构：')
print('  try:      可能出错的代码')
print('  except:   错误发生时执行')
print('  else:     没有错误时执行')
print('  finally:  无论是否出错都执行（清理工作）')
print()

def safe_divide(a, b):
    """安全的除法函数 - 演示完整的异常处理"""
    try:
        result = a / b
    except ZeroDivisionError:
        print(f'  [EXCEPT] 错误：{a}/{b} - 除数不能为零！')
        return None
    except TypeError:
        print(f'  [EXCEPT] 错误：{a}/{b} - 参数类型不正确！')
        return None
    else:
        print(f'  [ELSE] 计算成功：{a}/{b} = {result}')
        return result
    finally:
        print(f'  [FINALLY] safe_divide({a}, {b}) 调用完毕')

print('--- 测试用例 ---')
print()
safe_divide(10, 3)
print()
safe_divide(10, 0)
print()
safe_divide(10, 'abc')
print()

# ============ 实际应用：安全的用户输入 ============
print('--- 实际应用：安全的输入验证 ---')
print()

def get_valid_number(prompt, min_val=None, max_val=None):
    """获取有效的数字输入（带验证）"""
    # 模拟用户输入
    test_inputs = ['abc', '-5', '150', '85']
    
    for test_input in test_inputs:
        print(f'  用户输入: "{test_input}"')
        try:
            value = float(test_input)
            if min_val is not None and value < min_val:
                raise ValueError(f'值必须 >= {min_val}')
            if max_val is not None and value > max_val:
                raise ValueError(f'值必须 <= {max_val}')
            print(f'  -> 有效输入: {value}')
        except ValueError as e:
            print(f'  -> 无效输入: {e}')
        print()



In [ ]:
# 可视化：Python异常层次结构
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Python异常层次结构 (Exception Hierarchy)', fontsize=15, fontweight='bold', pad=15)

def draw_box(ax, x, y, text, color, w=2.2, h=0.5):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
        facecolor=color, edgecolor='black', linewidth=1.2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=8, fontweight='bold')
    return (x + w/2, y, x + w/2, y + h)  # bottom_center, top_center

# BaseException
draw_box(ax, 5.5, 9, 'BaseException', '#F44336', w=3, h=0.6)

# Exception
draw_box(ax, 5.9, 7.8, 'Exception', '#FF5722', w=2.5, h=0.5)
ax.plot([7.15, 7.0], [8.3, 9.0], 'k-', lw=1)

# 常见异常
exceptions = [
    (0.3, 6.2, 'ValueError\n值错误', '#FF9800'),
    (2.8, 6.2, 'TypeError\n类型错误', '#FFC107'),
    (5.3, 6.2, 'IndexError\n索引错误', '#4CAF50'),
    (7.8, 6.2, 'KeyError\n键错误', '#2196F3'),
    (10.3, 6.2, 'FileNotFoundError\n文件未找到', '#9C27B0'),
]

for x, y, text, color in exceptions:
    draw_box(ax, x, y, text, color, h=0.7)
    ax.plot([x+1.1, 7.15], [y+0.7, 7.8], 'k-', lw=0.8)

# 更多异常
more = [
    (0.5, 4.5, 'ZeroDivisionError\n除以零', '#FFB74D'),
    (3, 4.5, 'AttributeError\n属性错误', '#FFD54F'),
    (5.5, 4.5, 'NameError\n名称错误', '#81C784'),
    (8, 4.5, 'IOError\n输入输出错误', '#64B5F6'),
    (10.5, 4.5, 'RuntimeError\n运行时错误', '#CE93D8'),
]

for x, y, text, color in more:
    draw_box(ax, x, y, text, color, h=0.7)

# 使用场景说明
scenarios = [
    (1.4, 3.2, 'int("abc")', '#666'),
    (3.5, 3.2, '"3" + 5', '#666'),
    (6.4, 3.2, 'list[999]', '#666'),
    (8.9, 3.2, 'dict["???"]', '#666'),
    (11.4, 3.2, 'open("???.txt")', '#666'),
]

for x, y, text, color in scenarios:
    ax.text(x, y, text, ha='center', fontsize=7, family='monospace', color=color,
        bbox=dict(facecolor='lightyellow', edgecolor='gray', boxstyle='round,pad=0.2'))

# 底部说明
ax.text(7, 1.5, 'try-except 可以捕获特定类型的异常\n也可以用 except Exception 捕获所有异常（不推荐，因为会隐藏bug）\n最佳实践：只捕获你知道如何处理的异常',
    ha='center', fontsize=10,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5))

plt.tight_layout()


In [ ]:
# 综合示例：带异常处理的文件操作和数据处理
import tempfile, os

print('=== 综合示例：安全的成绩管理系统 ===')
print()

class GradeManager:
    """成绩管理器 - 综合文件处理和异常处理"""
    
    def __init__(self, filepath):
        self.filepath = filepath
        self.students = {}
    
    def load_from_file(self):
        """从文件加载成绩"""
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                header = f.readline()  # 跳过表头
                for line_num, line in enumerate(f, start=2):
                    try:
                        parts = line.strip().split(',')
                        if len(parts) != 3:
                            raise ValueError(f'格式错误：应有3个字段，实际有{len(parts)}个')
                        name = parts[0]
                        score = float(parts[1])
                        if not 0 <= score <= 100:
                            raise ValueError(f'成绩{score}超出范围(0-100)')
                        grade = parts[2]
                        self.students[name] = {'score': score, 'grade': grade}
                    except ValueError as e:
                        print(f'  警告：第{line_num}行数据有误 - {e}')
                        continue
            print(f'  成功加载 {len(self.students)} 条记录')
            return True
        except FileNotFoundError:
            print(f'  错误：文件 {self.filepath} 不存在')
            return False
        except PermissionError:
            print(f'  错误：没有权限读取文件')
            return False
    
    def save_to_file(self):
        """保存成绩到文件"""
        try:
            with open(self.filepath, 'w', encoding='utf-8') as f:
                f.write('姓名,成绩,等级\n')
                for name, data in self.students.items():
                    f.write(f'{name},{data["score"]},{data["grade"]}\n')
            print(f'  成功保存 {len(self.students)} 条记录')
            return True
        except IOError as e:
            print(f'  错误：保存失败 - {e}')
            return False
    
    def add_student(self, name, score):
        """添加学生成绩"""
        try:
            score = float(score)
            if not 0 <= score <= 100:
                raise ValueError('成绩必须在0-100之间')
            
            if score >= 90: grade = 'A*'
            elif score >= 80: grade = 'A'
            elif score >= 70: grade = 'B'
            elif score >= 60: grade = 'C'
            elif score >= 50: grade = 'D'
            else: grade = 'U'
            
            self.students[name] = {'score': score, 'grade': grade}
            print(f'  添加: {name}, {score}分, 等级{grade}')
        except (ValueError, TypeError) as e:
            print(f'  添加失败: {e}')
    
    def get_statistics(self):
        """统计分析"""
        if not self.students:
            print('  没有数据！')
            return
        scores = [s['score'] for s in self.students.values()]
        print(f'  学生人数: {len(scores)}')
        print(f'  平均分: {sum(scores)/len(scores):.1f}')
        print(f'  最高分: {max(scores)}')
        print(f'  最低分: {min(scores)}')


# ============ 演示 ============
temp_dir = tempfile.mkdtemp()
filepath = os.path.join(temp_dir, 'grades.csv')

manager = GradeManager(filepath)

# 尝试加载不存在的文件
print('1. 尝试加载不存在的文件：')
manager.load_from_file()
print()

# 添加学生
print('2. 添加学生：')
manager.add_student('张三', 92)
manager.add_student('李四', 78)
manager.add_student('王五', 'abc')  # 无效输入
manager.add_student('赵六', 150)    # 超出范围
manager.add_student('钱七', 85)
print()

# 保存
print('3. 保存到文件：')
manager.save_to_file()
print()

# 重新加载
print('4. 重新加载文件：')
manager2 = GradeManager(filepath)
manager2.load_from_file()
print()

# 统计
print('5. 统计分析：')
manager2.get_statistics()

import shutil


---
## 6. 函数式编程简介 (Functional Programming Introduction)

### 6.1 什么是函数式编程？

**函数式编程 (Functional Programming, FP)** 是声明式编程的另一个重要分支。它的核心思想：

1. **纯函数 (Pure Functions)**：同样的输入总是产生同样的输出，没有副作用
2. **不可变数据 (Immutable Data)**：不修改已有数据，而是创建新数据
3. **函数是一等公民 (First-class Functions)**：函数可以作为参数传递、作为返回值
4. **高阶函数 (Higher-order Functions)**：接受函数作为参数的函数

### 6.2 为什么函数式编程越来越重要？

1. **并发安全**：没有共享可变状态，天然适合多线程
2. **易于测试**：纯函数的行为完全由输入决定
3. **代码简洁**：map, filter, reduce 比循环更简洁
4. **数据处理**：大数据处理（如MapReduce）就是函数式思想

In [ ]:
# 函数式编程在Python中的应用
from functools import reduce

print('=== 函数式编程 (Functional Programming) 在Python中的应用 ===')
print()

# ============ 1. 纯函数 vs 非纯函数 ============
print('--- 1. 纯函数 vs 非纯函数 ---')
print()

# 非纯函数：有副作用
total = 0
def add_to_total(x):
    global total
    total += x  # 副作用：修改了外部变量
    return total

print('非纯函数（有副作用）：')
print(f'  add_to_total(5) = {add_to_total(5)}')
print(f'  add_to_total(5) = {add_to_total(5)}')  # 同样输入，不同输出！
print()

# 纯函数：无副作用
def pure_add(a, b):
    return a + b  # 无副作用，同样输入总是同样输出

print('纯函数（无副作用）：')
print(f'  pure_add(3, 5) = {pure_add(3, 5)}')
print(f'  pure_add(3, 5) = {pure_add(3, 5)}')  # 同样输入，同样输出
print()

# ============ 2. map, filter, reduce ============
print('--- 2. 高阶函数：map, filter, reduce ---')
print()

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f'原始列表: {numbers}')
print()

# map：对每个元素应用函数
squared = list(map(lambda x: x**2, numbers))
print(f'map(x -> x², numbers) = {squared}')
print('  map的作用："映射"每个元素到新值')
print()

# filter：过滤元素
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(f'filter(x -> x%2==0, numbers) = {evens}')
print('  filter的作用：只保留满足条件的元素')
print()

# reduce：将列表归约为单个值
total = reduce(lambda acc, x: acc + x, numbers)
print(f'reduce((acc, x) -> acc + x, numbers) = {total}')
print('  reduce的作用：将列表"折叠"为单个值')
print()

# ============ 3. 函数组合 ============
print('--- 3. 函数组合：流水线处理 ---')
print()
print('任务：取1-10中的偶数，求平方，再求和')
print()

# 命令式
result_imp = 0
for x in range(1, 11):
    if x % 2 == 0:
        result_imp += x ** 2
print(f'命令式: {result_imp}')

# 函数式
result_fp = reduce(
    lambda acc, x: acc + x,
    map(lambda x: x**2,
        filter(lambda x: x % 2 == 0, range(1, 11))))
print(f'函数式: {result_fp}')

# Python风格（列表推导）
result_py = sum(x**2 for x in range(1, 11) if x % 2 == 0)
print(f'Pythonic: {result_py}')
print()


In [ ]:
# 可视化：三种编程范式对比 - 处理数据的思维方式
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

task = '任务：[1,2,3,4,5,6,7,8,9,10] -> 取偶数 -> 平方 -> 求和 = 220'
fig.suptitle(f'三种范式处理数据的思维方式\n{task}', fontsize=13, fontweight='bold', y=0.98)

# === 命令式 ===
ax = axes[0]
ax.set_xlim(0, 14)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('命令式 (Imperative): 逐步修改状态', fontsize=11, fontweight='bold', color='#D32F2F', loc='left')

steps_imp = ['total=0', 'i=2: total=4', 'i=4: total=20', 'i=6: total=56', 'i=8: total=120', 'i=10: total=220']
for i, step in enumerate(steps_imp):
    x = 0.5 + i * 2.3
    box = mpatches.FancyBboxPatch((x, 0.5), 2, 0.8, boxstyle='round,pad=0.1',
        facecolor='#FFCDD2', edgecolor='#D32F2F', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x+1, 0.9, step, ha='center', va='center', fontsize=8, fontweight='bold')
    if i < len(steps_imp) - 1:
        ax.annotate('', xy=(x+2.1, 0.9), xytext=(x+2, 0.9),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=1.5))

# === 函数式 ===
ax = axes[1]
ax.set_xlim(0, 14)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('函数式 (Functional): 数据流经函数管道', fontsize=11, fontweight='bold', color='#2E7D32', loc='left')

pipeline = [
    ('[1..10]', '#C8E6C9'),
    ('filter(偶数)', '#A5D6A7'),
    ('[2,4,6,8,10]', '#81C784'),
    ('map(x²)', '#66BB6A'),
    ('[4,16,36,64,100]', '#4CAF50'),
    ('reduce(+)', '#388E3C'),
    ('220', '#2E7D32'),
]

for i, (text, color) in enumerate(pipeline):
    x = 0.3 + i * 1.9
    box = mpatches.FancyBboxPatch((x, 0.5), 1.7, 0.8, boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='#1B5E20', linewidth=1.2)
    ax.add_patch(box)
    fc = 'white' if i >= 5 else 'black'
    ax.text(x+0.85, 0.9, text, ha='center', va='center', fontsize=8, fontweight='bold', color=fc)
    if i < len(pipeline) - 1:
        ax.annotate('', xy=(x+1.8, 0.9), xytext=(x+1.7, 0.9),
            arrowprops=dict(arrowstyle='->', color='#1B5E20', lw=1.5))

# === 声明式 ===
ax = axes[2]
ax.set_xlim(0, 14)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('声明式 (Declarative): 只描述"要什么"', fontsize=11, fontweight='bold', color='#1565C0', loc='left')

box = mpatches.FancyBboxPatch((1, 0.3), 12, 1.2, boxstyle='round,pad=0.15',
    facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=2.5)
ax.add_patch(box)
ax.text(7, 1.0, 'SELECT SUM(x*x) FROM numbers WHERE x BETWEEN 1 AND 10 AND x % 2 = 0',
    ha='center', va='center', fontsize=10, fontweight='bold', family='monospace', color='#0D47A1')
ax.text(7, 0.5, '数据库引擎自动优化执行方案，程序员只描述需求',
    ha='center', va='center', fontsize=9, color='#1565C0')

plt.tight_layout(rect=[0, 0, 1, 0.95])


---
## 7. 练习题 (Exercises)

### 练习1：概念题

1. 声明式编程和命令式编程的核心区别是什么？用你自己的话解释。
2. SQL为什么是声明式语言？举一个SQL查询的例子来说明。
3. 解释Prolog中"事实"(Fact)、"规则"(Rule)和"查询"(Query)的区别。
4. 什么是"纯函数"(Pure Function)？为什么纯函数更容易测试？
5. 为什么异常处理很重要？不处理异常会怎样？

### 练习2：异常处理代码分析

```python
try:
    x = int(input("Enter a number: "))
    result = 100 / x
    print(f"Result: {result}")
except ValueError:
    print("Not a valid number")
except ZeroDivisionError:
    print("Cannot divide by zero")
finally:
    print("Done")
```

a) 如果用户输入 "5"，输出是什么？
b) 如果用户输入 "0"，输出是什么？
c) 如果用户输入 "abc"，输出是什么？
d) `finally` 块在什么情况下会执行？

### 练习3：编程实践

In [ ]:
# 练习3A：文件处理
# 任务：创建一个简单的"电话簿"程序
#
# 要求：
# 1. 创建函数 save_contacts(filepath, contacts) - 将联系人保存到文件
#    contacts是字典 {'张三': '13800001111', ...}
# 2. 创建函数 load_contacts(filepath) - 从文件加载联系人
# 3. 所有文件操作都要有异常处理
# 4. 创建函数 search_contact(contacts, name) - 搜索联系人

# 你的代码：
import tempfile, os

def save_contacts(filepath, contacts):
    # TODO: 实现文件写入，带异常处理
    pass

def load_contacts(filepath):
    # TODO: 实现文件读取，带异常处理
    pass

def search_contact(contacts, name):
    # TODO: 搜索联系人，带异常处理


In [ ]:
# 练习3B：函数式编程
# 任务：用map, filter, reduce完成以下操作
#
# 给定一个学生成绩列表：
# students = [('张三', 92), ('李四', 45), ('王五', 78), ('赵六', 88), ('钱七', 33)]
#
# 1. 用filter找出所有及格(>=60)的学生
# 2. 用map将及格学生的成绩转换为等级(>=90: A, >=80: B, >=70: C, >=60: D)
# 3. 用reduce计算所有学生的总分
# 4. 最终输出：及格学生及其等级、总分、平均分

from functools import reduce

students = [('张三', 92), ('李四', 45), ('王五', 78), ('赵六', 88), ('钱七', 33)]

# 你的代码：
# 1. filter
# passed = ...

# 2. map
# graded = ...

# 3. reduce


In [ ]:
# 练习4（挑战题）：用Python实现一个简易的Prolog引擎
#
# 扩展本节的PrologEngine类，添加以下功能：
# 1. 支持查询所有满足条件的结果（如：谁是张三的父母？）
# 2. 添加 uncle/aunt 规则
# 3. 添加 cousin 规则

# 提示：
# uncle(X, Y) :- sibling(X, Z), parent(Z, Y), male(X).
# cousin(X, Y) :- parent(A, X), parent(B, Y), sibling(A, B).

# 你的代码：


---
## 8. 本节小结 (Summary)

### 核心知识点

| 概念 | 英文 | 关键理解 |
|------|------|----------|
| 声明式编程 | Declarative Programming | 描述"要什么"而非"怎么做" |
| SQL | Structured Query Language | 最广泛的声明式语言，操作数据库 |
| Prolog | Programming in Logic | 基于事实、规则、查询的逻辑式编程 |
| 事实 | Facts | Prolog中的已知真命题 |
| 规则 | Rules | Prolog中的推理规则 |
| 查询 | Queries | Prolog中要回答的问题 |
| 函数式编程 | Functional Programming | 基于纯函数和不可变数据 |
| 异常处理 | Exception Handling | 优雅地处理运行时错误 |
| TRY...EXCEPT | Try-Except | 捕获和处理异常的结构 |
| 文件处理 | File Handling | 读写文件实现数据持久化 |

### 考试重点 (Exam Focus)

- 能区分声明式和命令式编程
- 能读懂简单的SQL查询和Prolog代码
- 能写出正确的 TRY...EXCEPT 异常处理代码
- 能进行基本的文件读写操作
- 理解 seek 和 tell 在随机文件访问中的作用

### 第20章总结

本章我们学习了多种编程范式：
1. **过程式** - 按步骤指令
2. **面向对象** - 用对象建模
3. **声明式** - 描述需求
4. **函数式** - 函数组合

每种范式都有其适用场景，优秀的程序员应该掌握多种范式，根据问题选择最合适的工具。